## Langchain Memory

1. Buffer memory: full history stored in memory
2. Window memory: Keep last N turns
3. Summary memory: summarize old turns automatically

In [39]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document

In [40]:
llm = OllamaLLM(model="llama3.1:8b")
embeddings = OllamaEmbeddings(model="llama3.1:8b")

In [41]:
loader = TextLoader("../data/Sandro_Botticelli.txt")
raw_docs = loader.load()

In [42]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
split_docs = splitter.split_documents(raw_docs)

In [43]:
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [44]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## No memory

In [45]:
# Basic prompt — no memory
baseline_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

In [46]:
baseline_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} 
    | baseline_prompt | llm | StrOutputParser()
)

In [47]:
print("Q: Who was Botticelli?")
print("A:", baseline_chain.invoke("Who was Botticelli?"))
print()
print("Q: Where did he live?")
print("A:", baseline_chain.invoke("Where did he live?"))

Q: Who was Botticelli?
A: Alessandro di Mariano di Vanni Filipepi, also known as Sandro Botticelli or simply Botticelli, was an Italian painter of the Early Renaissance.

Q: Where did he live?
A: I don't know.


## Buffer memory (stores full history)

In [48]:
# Prompt that includes chat history
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the question based ONLY on the following context.
    If the answer is not in the context, say "I don't know".
    
    Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])


In [49]:
# Chain with memory
memory_chain = (
    {
        "context": RunnableLambda(lambda x: x["question"]) | retriever | format_docs,
        "chat_history": RunnableLambda(lambda x: x["chat_history"]),
        "question": RunnableLambda(lambda x: x["question"]),
    }
    | memory_prompt
    | llm
    | StrOutputParser()
)

In [50]:
chat_history = []

def chat(question):
    global chat_history
    
    response = memory_chain.invoke({
        "question": question,
        "chat_history": chat_history,
    })
    
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    print(f"You: {question}")
    print(f"Bot: {response}")
    print(f"[{len(chat_history)//2} turn(s) in history]\n")
    
    return response

In [51]:
chat_history = []
chat("Who was Botticelli?")
chat("Where did he live?")
chat("What were his most famous paintings?")
chat("Which museum are they in?")

You: Who was Botticelli?
Bot: Botticelli, whose full name is Alessandro di Mariano di Vanni Filipepi, was an Italian painter of the Early Renaissance. He was born around 1445 and died on May 17, 1510.
[1 turn(s) in history]

You: Where did he live?
Bot: I don't know.
[2 turn(s) in history]

You: What were his most famous paintings?
Bot: Botticelli's posthumous reputation suffered until the late 19th century, but since then, his paintings have been seen to represent the linear grace of late Italian Gothic and some Early Renaissance painting. His four panels with Scenes from the Life of Saint Zenobius are notable examples of his later work, which features expressively distorted figures and a non-naturalistic use of colour.
[3 turn(s) in history]

You: Which museum are they in?
Bot: I don't know. The text doesn't mention where Botticelli's paintings are currently located. However, it mentions that the National Gallery bought Botticelli's Venus and Mars for £1,050 in 1874.
[4 turn(s) in hi

"I don't know. The text doesn't mention where Botticelli's paintings are currently located. However, it mentions that the National Gallery bought Botticelli's Venus and Mars for £1,050 in 1874."

### Rewrite prompt

Rewrite looks at the chat history then give the rewritten version of the question.

In [52]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given a conversation history and a follow-up question,
rewrite the follow-up question into a standalone question that contains
all necessary context from the history.

Return ONLY the rewritten question, nothing else. No explanation."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Follow-up question: {question}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

In [53]:
# Test rewriter in isolation
test_history = [
    HumanMessage(content="Who was Botticelli?"),
    AIMessage(content="Botticelli was an Italian painter of the Early Renaissance."),
]

rewritten = rewrite_chain.invoke({
    "chat_history": test_history,
    "question": "Where did he live?"
})
print("Original : Where did he live?")
print("Rewritten:", rewritten)

Original : Where did he live?
Rewritten: Who is the Italian painter mentioned in this conversation who painted during the Early Renaissance, and where did he reside?


### Add rewrite to full chain

In [54]:
def chat_with_rewrite(question):
    global chat_history

    # Step 1 — rewrite the question if there's history
    if chat_history:
        standalone_question = rewrite_chain.invoke({
            "chat_history": chat_history,
            "question": question,
        })
    else:
        standalone_question = question

    # Step 2 — retrieve using the rewritten question
    response = memory_chain.invoke({
        "question": standalone_question,
        "chat_history": chat_history,
    })

    # Step 3 — store original question in history (not rewritten)
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))

    print(f"You:      {question}")
    if chat_history:
        print(f"Rewritten: {standalone_question}")
    print(f"Bot:      {response}")
    print(f"[{len(chat_history)//2} turn(s) in history]\n")

    return response

In [55]:
chat_history = []
chat_with_rewrite("Who was Botticelli?")
chat_with_rewrite("Where did he live?")
chat_with_rewrite("What were his most famous paintings?")
chat_with_rewrite("Which museum are they in?")

You:      Who was Botticelli?
Rewritten: Who was Botticelli?
Bot:      Botticelli, whose full name is Alessandro di Mariano di Vanni Filipepi, was an Italian painter of the Early Renaissance. He is also known as Sandro Botticelli or simply Botticelli.
[1 turn(s) in history]

You:      Where did he live?
Rewritten: Where did Alessandro di Mariano di Vanni Filipepi, a.k.a. Botticelli, an Italian painter of the Early Renaissance, live?
Bot:      I don't know. The text doesn't mention where Botticelli lived. It mentions that he was asked to join a project in Pisa, but it doesn't provide information about his residence or hometown.
[2 turn(s) in history]

You:      What were his most famous paintings?
Rewritten: What are the notable works of Alessandro di Mariano di Vanni Filipepi, also known as Sandro Botticelli or simply Botticelli, an Italian painter of the Early Renaissance?
Bot:      According to the context, Botticelli's paintings represent the linear grace of late Italian Gothic and 

'According to the text, no specific painting is mentioned that is related to Dante Alighieri and is located in a particular museum. However, it does mention that Botticelli\'s "Venus and Mars" was bought by the National Gallery for £1,050 in 1874. \n\nSo, the answer would be: The Venus and Mars can be found in the National Gallery.'

In [56]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rewrite the follow-up question as a short, simple standalone question.
Use only the name or key term from history — no descriptions, no full names, no extra context.
Maximum 10 words. Return ONLY the rewritten question, nothing else."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Follow-up question: {question}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# Test
test_history = [
    HumanMessage(content="Who was Botticelli?"),
    AIMessage(content="Botticelli was an Italian painter of the Early Renaissance."),
]

rewritten = rewrite_chain.invoke({
    "chat_history": test_history,
    "question": "Where did he live?"
})
print("Original : Where did he live?")
print("Rewritten:", rewritten)

Original : Where did he live?
Rewritten: Where did Botticelli live?


In [57]:
chat_history = []
chat_with_rewrite("Who was Botticelli?")
chat_with_rewrite("Where did he live?")
chat_with_rewrite("What were his most famous paintings?")
chat_with_rewrite("Which museum are they in?")

You:      Who was Botticelli?
Rewritten: Who was Botticelli?
Bot:      Botticelli, whose full name is Alessandro di Mariano di Vanni Filipepi (1445 - 1510), was an Italian painter of the Early Renaissance. He is known for his paintings that represent the linear grace of late Italian Gothic and some Early Renaissance painting. His famous works include "The Birth of Venus" and "Primavera", both located in the Uffizi in Florence.
[1 turn(s) in history]

You:      Where did he live?
Rewritten: Where did Botticelli live?
Bot:      Botticelli lived all his life in the same neighbourhood of Florence, specifically the Ognissanti neighbourhood. He also spent some time painting in Pisa (in 1474) and Rome (1481-82), where he worked on a project for the Sistine Chapel, but Florence was his primary residence throughout his life.
[2 turn(s) in history]

You:      What were his most famous paintings?
Rewritten: What are Botticelli's most famous works?
Bot:      Botticelli's best-known works are "The 

'The Uffizi in Florence.'

## Window memory

The buffer memory keeps the entire history forever, meaning massive context passed to LLM every time. It is slower, more expensive and will hit context window limit 

In [59]:
def chat_with_window(question, window_size=3):
    global chat_history

    # Keep only last N turns
    windowed_history = chat_history[-(window_size * 2):]

    # Rewrite using windowed history
    if windowed_history:
        standalone_question = rewrite_chain.invoke({
            "chat_history": windowed_history,
            "question": question,
        })
    else:
        standalone_question = question

    response = memory_chain.invoke({
        "question": standalone_question,
        "chat_history": windowed_history,
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))

    print(f"You:       {question}")
    print(f"Rewritten: {standalone_question}")
    print(f"Bot:       {response}")
    print(f"[Total history: {len(chat_history)//2} turns | Window: {len(windowed_history)//2} turns]\n")

    return response


In [63]:
chat_history = []
chat_with_window("Who was Botticelli?")
chat_with_window("Where did he live?")
chat_with_window("What were his most famous paintings?")
chat_with_window("Which museum are they in?")
chat_with_window("When was he born?")
chat_with_window("Who is his wife?")

You:       Who was Botticelli?
Rewritten: Who was Botticelli?
Bot:       Botticelli, whose full name was Alessandro di Mariano di Vanni Filipepi (c. 1445 – May 17, 1510), was an Italian painter of the Early Renaissance.
[Total history: 1 turns | Window: 0 turns]

You:       Where did he live?
Rewritten: Where was Botticelli from?
Bot:       I don't know. The text does not mention the location of Botticelli's birth or where he is originally from. It only mentions that he lived all his life in a neighbourhood of Florence, and had some periods spent painting elsewhere (in Pisa and Rome).
[Total history: 2 turns | Window: 1 turns]

You:       What were his most famous paintings?
Rewritten: What were Botticelli's most famous works?
Bot:       Botticelli's best-known works are The Birth of Venus and Primavera, both located in the Uffizi in Florence.
[Total history: 3 turns | Window: 2 turns]

You:       Which museum are they in?
Rewritten: Which museum do his most famous paintings hang in?
B

"I don't know."

## Summary memory

Instead of cutting off old turns with window memory, we can summarize them. That way nothing is lost, but the context stays compact.

```
Full history (20 turns)
        ↓
[Turn 1...18 → compressed into 2-3 sentence summary]
[Turn 19-20 → kept as-is (recent turns)]
        ↓
LLM receives: summary + last 2 turns (not all 20)
```

In [64]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """Summarize the following conversation in 2-3 sentences.
Capture only the key facts discussed. Be concise.
Return ONLY the summary, nothing else."""),
    ("human", "{history}"),
])

summary_chain = summary_prompt | llm | StrOutputParser()

In [65]:
conversation_summary = ""  # grows over time

In [66]:
def chat_with_summary(question, keep_last_n=2):
    global chat_history, conversation_summary

    # Split history
    recent_turns = chat_history[-(keep_last_n * 2):] # last 2 turns
    older_turns = chat_history[:-(keep_last_n * 2)]

    # Summarize older turns
    if older_turns:
        older_text = "\n".join([
            f"{'User' if isinstance(m, HumanMessage) else 'Bot'}: {m.content}"
            for m in older_turns
        ])
        conversation_summary = summary_chain.invoke({"history": older_text})

    # Build context: summary + recent turns
    summary_message = []
    if conversation_summary:
        summary_message = [HumanMessage(content=f"[Conversation so far: {conversation_summary}]")]
    full_context = summary_message + recent_turns

    # Rewrite using full context
    if full_context:
        standalone_question = rewrite_chain.invoke({
            "chat_history": full_context,
            "question": question,
        })
    else:
        standalone_question = question

    response = memory_chain.invoke({
        "question": standalone_question,
        "chat_history": full_context,
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))

    print(f"You:       {question}")
    print(f"Rewritten: {standalone_question}")
    print(f"Bot:       {response}")
    if conversation_summary:
        print(f"Summary:   {conversation_summary}")
    print(f"[Total: {len(chat_history)//2} turns | Recent: {keep_last_n} | Summary: {'yes' if conversation_summary else 'no'}]\n")

    return response

In [68]:
chat_history = []
conversation_summary = ""
chat_with_summary("Who was Botticelli?")
chat_with_summary("Where did he live?")
chat_with_summary("What were his most famous paintings?")
chat_with_summary("Which museum are they in?")
chat_with_summary("Who were his patrons?")
chat_with_summary("Were he married?")


You:       Who was Botticelli?
Rewritten: Who was Botticelli?
Bot:       Botticelli, whose full name is Alessandro di Mariano di Vanni Filipepi, was an Italian painter of the Early Renaissance. He is also known as Sandro Botticelli or simply Botticelli.
[Total: 1 turns | Recent: 2 | Summary: no]

You:       Where did he live?
Rewritten: Where did Botticelli live?
Bot:       Botticelli lived all his life in the same neighbourhood of Florence. Specifically, he lived in the Ognissanti neighbourhood. There were a couple of exceptions, where he spent some time painting elsewhere: Pisa in 1474 and Rome (specifically, the Sistine Chapel) from 1481-82.
[Total: 2 turns | Recent: 2 | Summary: no]

You:       What were his most famous paintings?
Rewritten: What are Botticelli's best known artworks?
Bot:       Botticelli's best-known works are "The Birth of Venus" and "Primavera", both located in the Uffizi in Florence.
[Total: 3 turns | Recent: 2 | Summary: no]

You:       Which museum are they i

"I don't know."

## Persistent memory

save/load chat history to disk so it survives between sessions.

In [69]:
import json
from pathlib import Path

In [70]:
HISTORY_FILE = Path("../data/chat_history.json")

In [71]:
def save_history(history):
    data = [
        {
            "role": "human" if isinstance(m, HumanMessage) else "ai",
            "content": m.content
        }
        for m in history
    ]
    with open(HISTORY_FILE, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(history)//2} turns to disk.")

In [72]:
def load_history():
    if not HISTORY_FILE.exists():
        print("No history file found — starting fresh.")
        return []
    with open(HISTORY_FILE, "r") as f:
        data = json.load(f)
    history = [
        HumanMessage(content=m["content"]) if m["role"] == "human"
        else AIMessage(content=m["content"])
        for m in data
    ]
    print(f"Loaded {len(history)//2} turns from disk.")
    return history

In [73]:
save_history(chat_history)

Saved 6 turns to disk.


### Reload and continue

In [78]:
# Simulate a fresh session
chat_history = load_history()

chat_with_summary("Who was William Young Ottley?")

Loaded 6 turns from disk.
You:       Who was William Young Ottley?
Rewritten: Who was William Young Ottley?
Bot:       William Young Ottley was an English collector who bought Botticelli's The Mystical Nativity in Italy and brought it to London in 1799.
Summary:   Botticelli was an Italian painter of the Early Renaissance, known as Sandro Botticelli or simply Botticelli. He lived in Florence's Ognissanti neighbourhood and created famous works "The Birth of Venus" and "Primavera", both located at the Uffizi museum.
[Total: 7 turns | Recent: 2 | Summary: yes]



"William Young Ottley was an English collector who bought Botticelli's The Mystical Nativity in Italy and brought it to London in 1799."

In [80]:
chat_with_summary("What did he do with the painting?")

You:       What did he do with the painting?
Rewritten: What happened to the painting that Ottley bought?
Bot:       When Ottley tried to sell the painting in 1811, no buyer could be found. After his death, it was purchased by William Fuller Maitland of Stansted, who allowed it to be exhibited at a major art exhibition in Manchester in 1857, where it was viewed by over a million people.
Summary:   Botticelli, an Italian painter of the Early Renaissance, lived in Florence and spent some time painting in Pisa and Rome. His most famous works are "The Birth of Venus" and "Primavera", located in the Uffizi museum. He was financially supported by the authorities in Pisa and had a connection with the wealthy Rucellai family.
[Total: 8 turns | Recent: 2 | Summary: yes]



'When Ottley tried to sell the painting in 1811, no buyer could be found. After his death, it was purchased by William Fuller Maitland of Stansted, who allowed it to be exhibited at a major art exhibition in Manchester in 1857, where it was viewed by over a million people.'

In [81]:
save_history(chat_history)

Saved 8 turns to disk.
